Code:You CAPSTONE PROJECT: Survivor Analysis

In [95]:
import pandas as pd 
import sqlite3

In [96]:
# read in data

# Do I need Season Summary?

df_castaway_details = pd.read_excel('survivoR.xlsx', sheet_name="Castaway Details")
df_castaways = pd.read_excel('survivoR.xlsx', sheet_name="Castaways")
df_advantage_details = pd.read_excel('survivoR.xlsx', sheet_name="Advantage Details")
df_advantage_movement = pd.read_excel('survivoR.xlsx', sheet_name="Advantage Movement")

# Data Wrangling

## Clean Castaway Details

DO I ACTUALLY NEED CASTAWAY DETAILS?

In [97]:
# Castaway Details info

df_castaway_details.info()
df_castaway_details.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1160 entries, 0 to 1159
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Castaway Id         1160 non-null   object        
 1   Full Name           1158 non-null   object        
 2   Full Name Detailed  1158 non-null   object        
 3   Castaway            1160 non-null   object        
 4   Last Name           1158 non-null   object        
 5   Date of Birth       881 non-null    datetime64[ns]
 6   Date of Death       17 non-null     datetime64[ns]
 7   Gender              1160 non-null   object        
 8   African             733 non-null    float64       
 9   Asian               733 non-null    float64       
 10  Latin American      733 non-null    float64       
 11  Native American     733 non-null    float64       
 12  Bipoc               733 non-null    float64       
 13  Lgbt                1158 non-null   float64     

,Castaway Id,Full Name,Full Name Detailed,Castaway,Last Name,Date of Birth,Date of Death,Gender,African,Asian,...,Bipoc,Lgbt,Personality Type,Occupation,Collar,Three Words,Hobbies,Pet Peeves,Race,Ethnicity
0,US0001,Sonja Christopher,Sonja Christopher,Sonja,Christopher,1937-01-28,2024-04-26,Female,0.0,0.0,...,0.0,1.0,ENFP,Musician,No collar,NaN,NaN,NaN,NaN,NaN
1,US0002,B.B. Andersen,B.B. Andersen,B.B.,Andersen,1936-01-18,2013-10-29,Male,0.0,0.0,...,0.0,0.0,ESTJ,Real Estate Developer,White collar,NaN,NaN,NaN,NaN,NaN
2,US0003,Stacey Stillman,Stacey Stillman,Stacey,Stillman,1972-08-11,NaT,Female,0.0,0.0,...,0.0,0.0,ENTJ,Attorney,White collar,NaN,NaN,NaN,NaN,NaN
3,US0004,Ramona Gray,Ramona Gray,Ramona,Gray,1971-01-20,NaT,Female,1.0,0.0,...,1.0,0.0,ISTJ,Biochemist/Chemist,White collar,NaN,NaN,NaN,Black,NaN
4,US0005,Dirk Been,Dirk Been,Dirk,Been,1976-06-15,NaT,Male,0.0,0.0,...,0.0,0.0,ISFP,Dairy Farmer,Blue collar,NaN,NaN,NaN,NaN,NaN


In [98]:
# drop columns from Castaway Details that aren't needed

df_castaway_details = df_castaway_details.drop(columns=['Full Name', 'Castaway', 'Last Name', 'Date of Death', 'Occupation', 'Collar', 'Three Words', 'Hobbies', 'Pet Peeves'])
print(df_castaway_details.columns)

Index(['Castaway Id', 'Full Name Detailed', 'Date of Birth', 'Gender',
       'African', 'Asian', 'Latin American', 'Native American', 'Bipoc',
       'Lgbt', 'Personality Type', 'Race', 'Ethnicity'],
      dtype='object')


In [99]:
#rename columns

df_castaway_details = df_castaway_details.rename(columns={'Castaway Id':'Castaway_Id', 'Full Name Detailed':'Full_Name_Detailed', 'Date of Birth':'DOB', 'African':'African_American', 'Latin American':'Latin_American', 'Native American':'Native_American', 'Personality Type':'Personality_Type'})
print(df_castaway_details.columns)

Index(['Castaway_Id', 'Full_Name_Detailed', 'DOB', 'Gender',
       'African_American', 'Asian', 'Latin_American', 'Native_American',
       'Bipoc', 'Lgbt', 'Personality_Type', 'Race', 'Ethnicity'],
      dtype='object')


The scope of this project is to analyze the seasons of the United States version of the show, so I am eliminating all rows pertaining to seasons from any other countries.

In [100]:
# drop rows for non-US seasons

df_castaway_details_US = df_castaway_details[df_castaway_details['Castaway_Id'].str.contains("US")]
df_castaway_details_US.info()

<class 'pandas.core.frame.DataFrame'>
Index: 733 entries, 0 to 732
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Castaway_Id         733 non-null    object        
 1   Full_Name_Detailed  733 non-null    object        
 2   DOB                 733 non-null    datetime64[ns]
 3   Gender              733 non-null    object        
 4   African_American    733 non-null    float64       
 5   Asian               733 non-null    float64       
 6   Latin_American      733 non-null    float64       
 7   Native_American     733 non-null    float64       
 8   Bipoc               733 non-null    float64       
 9   Lgbt                733 non-null    float64       
 10  Personality_Type    733 non-null    object        
 11  Race                175 non-null    object        
 12  Ethnicity           114 non-null    object        
dtypes: datetime64[ns](1), float64(6), object(6)
memory usag

The only columns in this data frame with null values are Race and Ethnicity.

To decide how to move forward with the null values, I referred back to the ReadMe.md document for the data source to determine if Anyone who did not have a race or ethnicity listed could be assumed to be white, since a lot of the data seems to be focused on identifying BIPOC individuals. The author of the data set states "If no source was found to determine a castaways race and ethnicity, the data is kept as missing rather than making an assumption." For this reason, I decided to drop the columns because they won't be useful for analysis.

In [101]:
# handle null values

df_castaway_details_US = df_castaway_details_US.drop(columns=['Race', 'Ethnicity'])

In [102]:
# Ensure all columns have the correct data type

df_castaway_details_US = df_castaway_details_US.astype({'Castaway_Id': 'str', 'Full_Name_Detailed': 'str', 'Gender': 'str', 'African_American':'bool', 'Asian':'bool', 'Latin_American': 'bool', 'Native_American':'bool', 'Bipoc':'bool', 'Lgbt':'bool', 'Personality_Type':'str'} )
print(df_castaway_details_US.dtypes)


Castaway_Id                   object
Full_Name_Detailed            object
DOB                   datetime64[ns]
Gender                        object
African_American                bool
Asian                           bool
Latin_American                  bool
Native_American                 bool
Bipoc                           bool
Lgbt                            bool
Personality_Type              object
dtype: object


In [103]:
# determine if Castaway Details contains any duplicates
print(df_castaway_details_US['Full_Name_Detailed'].describe())

count                   733
unique                  733
top       Sonja Christopher
freq                      1
Name: Full_Name_Detailed, dtype: object


The Castaway Details only contains one row per Castaway.

## Function to Drop non-US rows

In [104]:
#function to drop non-US rows from a dataframe
# use on Advantage Details, Advantage Movement, and Castaways

def drop_non_us (df: pd.DataFrame) -> pd.DataFrame:
    """Drops all rows from the table where the Version does not eqal 'US' """
    df_non_US_dropped = df[df['Version']=="US"]
    return df_non_US_dropped

## Clean Castaways

In [105]:
# Castaways info

df_castaways.info()
df_castaways.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1403 entries, 0 to 1402
Data columns (total 26 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Version         1403 non-null   object 
 1   Version Season  1403 non-null   object 
 2   Season          1403 non-null   int64  
 3   Full Name       1403 non-null   object 
 4   Castaway Id     1403 non-null   object 
 5   Castaway        1403 non-null   object 
 6   Age             1361 non-null   float64
 7   City            1349 non-null   object 
 8   State           1299 non-null   object 
 9   Episode         1361 non-null   float64
 10  Day             1361 non-null   float64
 11  Order           1361 non-null   float64
 12  Result          1361 non-null   object 
 13  Jury Status     601 non-null    object 
 14  Place           1403 non-null   int64  
 15  Original Tribe  1359 non-null   object 
 16  Jury            1403 non-null   bool   
 17  Finalist        1361 non-null   f

,Version,Version Season,Season,Full Name,Castaway Id,Castaway,Age,City,State,Episode,...,Jury,Finalist,Winner,Acknowledge,Ack Look,Ack Speak,Ack Gesture,Ack Smile,Ack Quote,Ack Score
0,US,US01,1,Sonja Christopher,US0001,Sonja,63.0,Walnut Creek,California,1.0,...,False,0.0,0.0,1.0,1.0,1.0,1.0,1.0,"""Go get 'em you guys.""",4.0
1,US,US01,1,B.B. Andersen,US0002,B.B.,64.0,Mission Hills,Kansas,2.0,...,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0
2,US,US01,1,Stacey Stillman,US0003,Stacey,27.0,San Francisco,California,3.0,...,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0
3,US,US01,1,Ramona Gray,US0004,Ramona,29.0,Edison,New Jersey,4.0,...,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0
4,US,US01,1,Dirk Been,US0005,Dirk,23.0,Spring Green,Wisconsin,5.0,...,False,0.0,0.0,1.0,1.0,1.0,1.0,1.0,"""Good luck""",4.0


In [106]:
# drop columns from Castaways that aren't needed

df_castaways = df_castaways.drop(columns=['Acknowledge', 'Ack Look', 'Ack Speak', 'Ack Gesture', 'Ack Smile', 'Ack Quote', 'Ack Score', 'Castaway'])
print(df_castaways.columns)

Index(['Version', 'Version Season', 'Season', 'Full Name', 'Castaway Id',
       'Age', 'City', 'State', 'Episode', 'Day', 'Order', 'Result',
       'Jury Status', 'Place', 'Original Tribe', 'Jury', 'Finalist', 'Winner'],
      dtype='object')


In [107]:
# rename columns

df_castaways = df_castaways.rename(columns={'Version Season':'Version_Season', 'Full Name':'Full_Name', 'Castaway Id':'Castaway_Id', 'Jury Status':'Jury_Status', 'Original Tribe':'Original_Tribe'})

In [108]:
# Ensure all columns have the correct data type

# should I treat season, etc. as a string? Will I use it to calculate anything or just categorize?

df_castaways= df_castaways.astype({'Version':'str', 'Version_Season':'str', 'Full_Name':'str', 'Castaway_Id':'str', 'City':'str', 'State':'str', 'Result':'str', 'Jury_Status':'str', 'Original_Tribe':'str', 'Finalist':'bool', 'Winner':'bool'})

In [109]:
print(df_castaways['Version'].unique())

['US' 'AU' 'SA' 'UK' 'NZ']


In [110]:
#Drop non-US seasons

df_castaways_US = drop_non_us(df_castaways_correct_data_type)
print(df_castaways_US['Version'].unique())

['US']


In [111]:
def finale_categorization (row: pd.Series) -> str:
        """ Takes the Jury, Finalist, and Winner columns and combine into one column that uses string instead of boolean """
        if row['Winner']:
              return 'Winner'
        elif row ['Finalist']:
              return 'Finalist'
        elif row['Jury']: 
            return 'Jury'
        else:
              return 'Voted out pre-jury'


df_castaways_US['Finale_Categorization'] = df_castaways_US.apply(finale_categorization, axis=1)

print(df_castaways_US)

    Version Version_Season  Season             Full_Name Castaway_Id   Age  \
0        US           US01       1     Sonja Christopher      US0001  63.0   
1        US           US01       1         B.B. Andersen      US0002  64.0   
2        US           US01       1       Stacey Stillman      US0003  27.0   
3        US           US01       1           Ramona Gray      US0004  29.0   
4        US           US01       1             Dirk Been      US0005  23.0   
..      ...            ...     ...                   ...         ...   ...   
912      US           US50      50           Rick Devens      US0560   NaN   
913      US           US50      50    Stephenie LaGrossa      US0144   NaN   
914      US           US50      50                  TBD1      US0734   NaN   
915      US           US50      50                  TBD2      US0735   NaN   
916      US           US50      50  Tiffany Nicole Ervin      US0695   NaN   

              City       State  Episode   Day  Order         Re

/var/folders/v5/6mwy7h_s1tqbsvd4zh8y744r0000gn/T/ipykernel_29594/1816899500.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_castaways_US['Finale_Categorization'] = df_castaways_US.apply(finale_categorization, axis=1)


Survivor is generally considered to have "eras" in which the production of the show and style of game play have themes that lend them to be grouped together and distinguished from other eras. Because advantages did not exist with in Survivor when it debut and the ways in which advantages have been incorporated into gameplay have evolved over the years, I am going to categorize each season into an era. 

There are is no official categorization of these eras. For the purposes of this project, they will be grouped in the following manner.


Old School (pre-advantage):1-10
Old School (with advantages): 11-20
New School: 21-40
New Era: 41 to present

In [112]:
# era categorization

def era_categorization (season: int) -> str: 
    """ Categorize a season into an era based on its number """
    if season <=10:
        return 'Old School (pre-advantage)'
    elif season <=20:
        return 'Old School (with advantages)'
    elif season <=40:
        return 'New School'
    elif season >40:
        return 'New Era'
    else:
        return 'No Era Assigned'

df_castaways_US['Era'] = df_castaways_US['Season'].apply(era_categorization)

print(df_castaways_US)

    Version Version_Season  Season             Full_Name Castaway_Id   Age  \
0        US           US01       1     Sonja Christopher      US0001  63.0   
1        US           US01       1         B.B. Andersen      US0002  64.0   
2        US           US01       1       Stacey Stillman      US0003  27.0   
3        US           US01       1           Ramona Gray      US0004  29.0   
4        US           US01       1             Dirk Been      US0005  23.0   
..      ...            ...     ...                   ...         ...   ...   
912      US           US50      50           Rick Devens      US0560   NaN   
913      US           US50      50    Stephenie LaGrossa      US0144   NaN   
914      US           US50      50                  TBD1      US0734   NaN   
915      US           US50      50                  TBD2      US0735   NaN   
916      US           US50      50  Tiffany Nicole Ervin      US0695   NaN   

              City       State  Episode   Day  Order         Re

/var/folders/v5/6mwy7h_s1tqbsvd4zh8y744r0000gn/T/ipykernel_29594/1508813439.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_castaways_US['Era'] = df_castaways_US['Season'].apply(era_categorization)


## Clean Advantage Details

In [113]:
df_advantage_details.info()
df_advantage_details.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Version         395 non-null    object
 1   Version Season  395 non-null    object
 2   Season          395 non-null    int64 
 3   Advantage Id    395 non-null    int64 
 4   Advantage Type  395 non-null    object
 5   Clue Details    395 non-null    object
 6   Location Found  387 non-null    object
 7   Conditions      194 non-null    object
dtypes: int64(2), object(6)
memory usage: 24.8+ KB


,Version,Version Season,Season,Advantage Id,Advantage Type,Clue Details,Location Found,Conditions
0,US,US11,11,1,Preventative Hidden Immunity Idol,Found without clue,Found around camp,Play before votes are cast
1,US,US12,12,1,Super Idol,Found on Exile,Advantage from Exile,Play after votes are read; Valid until F4
2,US,US13,13,1,Super Idol,Found on Exile,Advantage from Exile,Play after votes are read; Valid until F4
3,US,US14,14,1,Hidden Immunity Idol,Found on Exile,Found around camp,NaN
4,US,US14,14,2,Hidden Immunity Idol,Someone shared the clue,Found around camp,NaN


In [114]:
# Drop columns from Advantage Details that aren't needed

In [115]:
#rename columns

In [116]:
# Ensure all columns have the correct data type

## Clean Advantage Movement

In [117]:
#Advantage Movement info

df_advantage_movement.info()
df_advantage_movement.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 902 entries, 0 to 901
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Version          902 non-null    object 
 1   Version Season   902 non-null    object 
 2   Season           902 non-null    int64  
 3   Castaway         902 non-null    object 
 4   Castaway Id      902 non-null    object 
 5   Advantage Id     902 non-null    int64  
 6   Sequence Id      902 non-null    int64  
 7   Day              885 non-null    float64
 8   Episode          885 non-null    float64
 9   Event            902 non-null    object 
 10  Played for       274 non-null    object 
 11  Played for Id    273 non-null    object 
 12  Success          276 non-null    object 
 13  Votes Nullified  206 non-null    float64
 14  Sog Id           881 non-null    float64
dtypes: float64(4), int64(3), object(8)
memory usage: 105.8+ KB


,Version,Version Season,Season,Castaway,Castaway Id,Advantage Id,Sequence Id,Day,Episode,Event,Played for,Played for Id,Success,Votes Nullified,Sog Id
0,US,US11,11,Gary,US0161,1,1,24.0,9.0,Found,NaN,NaN,NaN,NaN,10.0
1,US,US11,11,Gary,US0161,1,2,24.0,9.0,Played,Gary,US0161,Yes,0.0,10.0
2,US,US12,12,Terry,US0180,1,1,9.0,4.0,Found,NaN,NaN,NaN,NaN,4.0
3,US,US12,12,Terry,US0180,1,2,37.0,15.0,Expired,NaN,NaN,NaN,NaN,13.0
4,US,US13,13,Yul,US0202,1,1,5.0,2.0,Found,NaN,NaN,NaN,NaN,2.0


In [118]:
#drop columns from Advantage Movement that aren't needed

df_advantage_movement.drop(columns=['Votes Nullified', 'Played for', 'Played for Id', 'Sog Id'])

,Version,Version Season,Season,Castaway,Castaway Id,Advantage Id,Sequence Id,Day,Episode,Event,Success
0,US,US11,11,Gary,US0161,1,1,24.0,9.0,Found,NaN
1,US,US11,11,Gary,US0161,1,2,24.0,9.0,Played,Yes
2,US,US12,12,Terry,US0180,1,1,9.0,4.0,Found,NaN
3,US,US12,12,Terry,US0180,1,2,37.0,15.0,Expired,NaN
4,US,US13,13,Yul,US0202,1,1,5.0,2.0,Found,NaN
...,...,...,...,...,...,...,...,...,...,...,...
897,NZ,NZ02,2,Eve,NZ0028,2,1,7.0,3.0,Found,NaN
898,NZ,NZ02,2,Eve,NZ0028,2,2,15.0,5.0,Played,No
899,NZ,NZ02,2,Josh,NZ0022,3,1,12.0,4.0,Activated,NaN
900,NZ,NZ02,2,Josh,NZ0022,3,2,13.0,5.0,Expired,NaN


In [119]:
#rename columns

In [120]:
# Ensure all columns have the correct data type